# Slippage analysis

Drop a SlippageOrdersReport `.xlsx` path into `FILE` below and run the cells top-to-bottom. Plotly charts render inline. Tables print inline.

Sections: headline · pareto · per-symbol · BC-vs-Maker divergence · distribution · worst individual trades · time-of-day · by taker · XAUUSD scatter.

In [ ]:
# ---- setup ----
FILE = 'slippageordersreport.xlsx'  # <-- set to your report path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt

pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

df = pd.read_excel(FILE, sheet_name='SlippageOrdersReport', header=1)
df = df.rename(columns={
    'Expected Price':'BC_Expected','Filled Price':'BC_Filled','Slippage':'BC_Slip','Slippage (USD)':'BC_SlipUSD',
    'Expected Price.1':'MK_Expected','Filled Price.1':'MK_Filled','Slippage.1':'MK_Slip','Slippage (USD).1':'MK_SlipUSD',
})
df['Recv Time'] = pd.to_datetime(df['Recv Time'])
df['hour_utc'] = df['Recv Time'].dt.hour
df['date'] = df['Recv Time'].dt.date

print(f'Loaded {len(df):,} orders')
print(f'Date range: {df["Recv Time"].min()}  →  {df["Recv Time"].max()}')
print(f'Total notional: ${df["Notional"].sum():,.0f}')
df.head()

## 1. Headline
One row per side (broker vs client-facing). Skew and kurtosis show whether the distribution is normal (both ~0) or tail-driven.

In [ ]:
rows = []
for lbl, col in [('BC (broker)','BC_SlipUSD'), ('Maker (client)','MK_SlipUSD')]:
    s = df[col]
    rows.append({
        'Side': lbl,
        'Sum $': s.sum(),
        'Orders': len(s),
        '% orders slipped': (s!=0).mean()*100,
        '% positive slip': (s>0).mean()*100,
        '% negative slip': (s<0).mean()*100,
        'Mean $': s.mean(),
        'Median $': s.median(),
        'Std $': s.std(),
        'Skew': s.skew(),
        'Excess kurt': s.kurt(),
        'Worst single $': s.min(),
    })
headline = pd.DataFrame(rows)
headline

## 2. Pareto — is it a few big orders or many small?

Table + cumulative-loss curve. If the curve elbows out quickly, a small number of orders drive the pain.

In [ ]:
pareto_rows = []
for lbl, col in [('BC','BC_SlipUSD'),('MK','MK_SlipUSD')]:
    neg = df[df[col]<0][col].sort_values()
    total = neg.sum()
    for pct in [1,5,10,25,50]:
        n = max(1, int(len(neg)*pct/100))
        pareto_rows.append({
            'Side': lbl,
            'Worst %': f'{pct}%',
            'N orders': n,
            'Sum $': neg.head(n).sum(),
            '% of total loss': neg.head(n).sum()/total*100 if total else 0,
        })
pareto = pd.DataFrame(pareto_rows)
pareto

In [ ]:
fig = go.Figure()
for lbl, col in [('BC (broker)','BC_SlipUSD'), ('Maker (client)','MK_SlipUSD')]:
    neg = df[df[col]<0][col].sort_values().values
    if len(neg)==0: continue
    x = np.arange(1, len(neg)+1)/len(neg)*100
    y = np.cumsum(neg)/neg.sum()*100
    fig.add_trace(go.Scatter(x=x, y=y, name=lbl, mode='lines'))
fig.update_layout(
    title='Pareto — cumulative slippage $ vs. worst-N% of slipped orders',
    xaxis_title='Worst orders (%)', yaxis_title='Cumulative slippage (% of total)',
    template='plotly_white')
fig.show()

## 3. Per-symbol view (normalised)

**bc_bps / mk_bps** = slippage as basis points of notional. Removes the 'higher-volume-symbols-slip-more' bias so you can compare symbols apples-to-apples.

**mk_minus_bc** = how much the desk absorbs that the broker slippage didn't cause.

In [ ]:
g = df.groupby('Symbol').agg(
    orders=('Client Ord Id','count'),
    notional=('Notional','sum'),
    bc_slip_usd=('BC_SlipUSD','sum'),
    mk_slip_usd=('MK_SlipUSD','sum'),
    pct_slipped_bc=('BC_SlipUSD', lambda s: (s!=0).mean()*100),
    pct_slipped_mk=('MK_SlipUSD', lambda s: (s!=0).mean()*100),
    worst_mk=('MK_SlipUSD','min'),
).reset_index()
g['bc_bps'] = g['bc_slip_usd']/g['notional']*10_000
g['mk_bps'] = g['mk_slip_usd']/g['notional']*10_000
g['mk_minus_bc'] = g['mk_slip_usd'] - g['bc_slip_usd']
g = g.sort_values('mk_slip_usd').reset_index(drop=True)
g.head(20)

In [ ]:
top = g.head(15)
fig = go.Figure()
fig.add_bar(x=top['Symbol'], y=top['bc_slip_usd'], name='BC (broker)', marker_color='steelblue')
fig.add_bar(x=top['Symbol'], y=top['mk_slip_usd'], name='Maker (client)', marker_color='crimson')
fig.update_layout(
    title='Top 15 symbols — BC vs Maker slippage ($ USD)',
    barmode='group', xaxis_title='', yaxis_title='Slippage USD',
    template='plotly_white')
fig.show()

In [ ]:
# bps view — normalised, spots symbols slipping way more than their peers per $ traded
top_bps = g.reindex(g['mk_bps'].abs().sort_values(ascending=False).index).head(15)
fig = px.bar(top_bps, x='Symbol', y='mk_bps',
             title='Worst 15 symbols by Maker slippage in bps of notional',
             template='plotly_white', color='mk_bps', color_continuous_scale='RdYlGn')
fig.show()

## 4. BC-vs-Maker divergence

Where the desk retains slippage the broker didn't cause. Big negative `mk_minus_bc` = markup pass-through leaking money.

In [ ]:
d = g.reindex(g['mk_minus_bc'].abs().sort_values(ascending=False).index).head(15)
d = d[['Symbol','orders','bc_slip_usd','mk_slip_usd','mk_minus_bc']].copy()
d['ratio_mk_over_bc'] = d['mk_slip_usd'] / d['bc_slip_usd'].replace(0, np.nan)
d

## 5. Distribution shape

Log-y histogram of |slippage $|. If it's a fat tail, you'll see a long thin bar way to the right.

In [ ]:
fig = go.Figure()
for lbl, col in [('BC','BC_SlipUSD'),('MK','MK_SlipUSD')]:
    s = df[df[col]!=0][col].abs()
    if len(s)==0: continue
    fig.add_trace(go.Histogram(x=s, name=lbl, nbinsx=80, opacity=0.6))
fig.update_layout(
    title='Distribution of |slippage $| (log-y) — tail visibility',
    xaxis_title='|Slippage USD|', yaxis_title='Order count',
    yaxis_type='log', barmode='overlay', template='plotly_white')
fig.show()

## 6. Worst individual trades

Because 10 trades account for ~13% of maker loss, they're worth eyeballing individually.

In [ ]:
cols = ['Recv Time','Taker','Symbol','Side','Fill Volume','Notional','BC_SlipUSD','MK_SlipUSD']
df.nsmallest(20, 'MK_SlipUSD')[cols]

## 7. Time of day

Rollover pain, session opens, thin-liquidity windows all live here.

In [ ]:
h = df.groupby('hour_utc').agg(
    orders=('Client Ord Id','count'),
    bc_slip_usd=('BC_SlipUSD','sum'),
    mk_slip_usd=('MK_SlipUSD','sum'),
    worst_mk=('MK_SlipUSD','min'),
).reset_index()
h

In [ ]:
fig = go.Figure()
fig.add_bar(x=h['hour_utc'], y=h['mk_slip_usd'], name='Maker $', marker_color='crimson')
fig.add_bar(x=h['hour_utc'], y=h['bc_slip_usd'], name='BC $', marker_color='steelblue')
fig.update_layout(
    title='Slippage by hour of day (UTC)',
    barmode='group', xaxis=dict(title='Hour UTC', dtick=1), yaxis_title='Slippage USD',
    template='plotly_white')
fig.show()

## 8. By taker

Does one venue leak more per $ traded than the other?

In [ ]:
t = df.groupby('Taker').agg(
    orders=('Client Ord Id','count'),
    notional=('Notional','sum'),
    bc_slip_usd=('BC_SlipUSD','sum'),
    mk_slip_usd=('MK_SlipUSD','sum'),
).reset_index()
t['mk_bps'] = t['mk_slip_usd']/t['notional']*10_000
t

## 9. XAUUSD scatter — volume vs slippage

Change `SYMBOL` below to look at any other name. If the cloud is randomly scattered, size isn't the driver — probably time or LP is.

In [ ]:
SYMBOL = 'XAUUSD'
sub = df[df['Symbol']==SYMBOL].copy()
r = sub['Notional'].corr(sub['MK_SlipUSD'].abs())
fig = px.scatter(
    sub, x='Notional', y='MK_SlipUSD',
    color='Side', hover_data=['Recv Time','Fill Volume'],
    title=f'{SYMBOL}: notional vs maker slippage $  (corr with |slip| = {r:.2f})',
    template='plotly_white')
fig.show()

## 10. Static PNG for pasting into a deck

Two-panel summary: top symbols + cumulative Pareto curve. Saved next to the notebook as `summary.png`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
top12 = g.head(12)
axes[0].barh(top12['Symbol'][::-1], top12['mk_slip_usd'][::-1], color='crimson')
axes[0].set_title('Top 12 symbols — Maker slippage $')
axes[0].set_xlabel('USD')
axes[0].grid(True, alpha=0.3)

neg = g.sort_values('mk_slip_usd')['mk_slip_usd'].values
cum = np.cumsum(neg)/neg.sum()*100
axes[1].plot(range(1, len(cum)+1), cum, marker='o', color='crimson')
axes[1].set_title('Cumulative Maker slippage vs symbols ranked worst→best')
axes[1].set_xlabel('Symbols (worst first)'); axes[1].set_ylabel('% of total loss')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('summary.png', dpi=150, bbox_inches='tight')
plt.show()